# Board A — interactive dashboard

## What this is
- **Left (~3/4):** synthetic index chart, **sector lanes**, and **per-symbol tiles** (dark, Robinhood-inspired).
- **Right (~1/4):** sector mix → parse → program FPGA over **AXI**; **regime** and **quote interval**; **Live poll (5 Hz)** to refresh the chart and tiles.

**Prices on screen:** *software synthetic* — they follow `QUOTES_SENT`, regime, and the mids you wrote, not exact RTL `best_bid`/`best_ask` (those are not on AXI today).

---

## Prerequisites
1. **Python packages:** `ipywidgets`, `anywidget` (needed by Plotly 6 for live charts), `plotly`, and on the board **`pynq`** (PYNQ images usually have it). From the repo root run:
   ```bash
   pip install -r sw/board_a_dashboard/requirements-dashboard.txt
   ```
2. **Repo layout:** this notebook lives in `sw/board_a_dashboard/`. The next cell adds `sw/` to `sys.path` so `board_a` and `board_a_dashboard` import correctly when you run from the repo root, from `sw/`, or from `sw/board_a_dashboard/`.

---

## Option A — Jupyter (recommended while developing)
1. Copy or sync the **`sw/`** tree onto the PYNQ (or open the repo from a shared drive).
2. Start Jupyter on the board, **browse to** `sw/board_a_dashboard/`, and open **`Board_A_Dashboard.ipynb`**.
3. **Run the import cell** (second cell). If `ModuleNotFoundError: board_a_dashboard`, use *Kernel → Change working directory* to the folder that contains **`sw`** as a subfolder, or open the notebook from inside `sw/board_a_dashboard/` and re-run.
4. In the **third cell**, set `USE_FPGA = True` if you have **`overlays/board_a.bit`** loaded on that image; keep `False` for a **demo** (no bitstream, fake MMIO).
5. **Run the third cell.** The dashboard appears below.
6. **Use the UI:** set **How many sectors**, pick each sector from the **short-name dropdowns**, enter counts for all rows **except the last** (the last row’s count fills in automatically to total 16 slots). Click **Parse & pick companies** → **Reset + write FPGA + start** → turn on **Live poll (5 Hz)**. Optional: **Advanced** accordion to paste a compact mix string instead. Adjust **Regime** / **Quote int.** anytime.

---

## Option B — Voilà (dashboard-only URL)
1. On the board, in a terminal, **`cd`** to the folder that contains this notebook (usually `.../sw/board_a_dashboard`).
2. Run:
   ```bash
   voila Board_A_Dashboard.ipynb
   ```
3. Open the URL Voilà prints (often port 8866). Set `USE_FPGA` in the notebook source if you need the real overlay, or edit the default before launching.

---

## Overlay path (PYNQ)
With `USE_FPGA = True`, the notebook uses **`overlays/board_a.bit`** (PYNQ default layout). If your `.bit` lives elsewhere, change that string in the third cell to match your project (e.g. full path under `/home/xilinx/...`).

In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
candidates = [here, here / "sw", here.parent, here.parent / "sw"]
for p in candidates:
    if (p / "board_a").is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

from board_a_dashboard import BoardADashboard, create_demo_dashboard, open_mmio_from_overlay, show

In [ ]:
# ---------------------------------------------------------------------------
# 1) USE_FPGA: True = real Board A overlay + AXI. False = DemoMMIO (laptop / no .bit).
# 2) Overlay path must match where you installed board_a.bit (default PYNQ layout).
# ---------------------------------------------------------------------------
USE_FPGA = False
OVERLAY_PATH = "overlays/board_a.bit"

if USE_FPGA:
    from pynq import Overlay

    ol = Overlay(OVERLAY_PATH)
    mmio = open_mmio_from_overlay(ol)
    dash = BoardADashboard(mmio)
else:
    dash = create_demo_dashboard()

show(dash)